# Earlier findings of amjad-2025-01, checked on version 0.261120.1306

The earlier attempt (amjad-2025-01) tested version 0.261012.0921 of
[DANDI:002013](https://dandiarchive.org/dandiset/002013/0.261120.1306). This notebook checks, on version
0.261120.1306, each of its findings that the blind attempt amjad-2025-02 did not report. Each section opens
with the earlier statement. Findings 3, 4, 11, 12, 14 and the cl3 and 8000 Hz parts of 13 are reported by
amjad-2025-02 itself and are not checked here. The loading helpers are those of `amjad.py`.

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal

import amjad as A

print("dandiset", A.DANDISET_ID, "version", A.DANDISET_VERSION)
F = {k: A.open_file(p) for k, p in A.PATHS.items()}
for k, f in F.items():
    print(k, A.PATHS[k], "session start", A._str(f["session_start_time"][()]))

## Finding 1

Earlier statement: "Each event is stored about five times in the events tables of both Monkey T sessions
(85665 stored times, 17686 distinct, in 09/13/2024; 59702 and 12434 in 09/26/2024), once for every
overlapping 60 s trial window that contains it."

In [ ]:
EV = {}
for k in ["shape", "direction", "monkey_p"]:
    EV[k], names = A.events(F[k])
    allt = np.concatenate(list(EV[k].values()))
    in_order = all(bool(np.all(np.diff(t) >= 0)) for t in EV[k].values())
    print(f"{k}: {len(allt)} stored times, {len(np.unique(allt))} distinct; repeated within a code: "
          f"{sum(len(t) - len(np.unique(t)) for t in EV[k].values())}; times in order within each code: {in_order}")

Each event is now stored once in both Monkey T sessions: fixed.

## Finding 2

Earlier statement: "The trials tables hold only times, no trial conditions. Conditions must be decoded
from the event codes. Session 09/13/2024 has no small-reward code, and the labels of codes 35 and 36
("right cue reward condition 1/2") do not say that they code large and small reward, which the data show
in all 303 successful forced trials."

In [ ]:
for k in ["shape", "direction"]:
    g = F[k]["intervals/trials"]
    print(f"{k}: trials columns {list(g.attrs['colnames'])}; distinct tags {sorted(set(A._str(x) for x in g['tags'][:]))}")
ev, names = A.events(F["shape"])
print("09/13/2024: code 100 present:", 100 in ev, "; labels:", {c: names[c] for c in (35, 36, 101)})
tr = A.decode_trials(ev, F["shape"]["intervals/trials/id"].shape[0])
succ = tr[tr.has_119 & tr.has_21]
print("successful forced trials:", len(succ))
print(pd.crosstab(succ.has_101.rename("big reward (101)"), [succ.has_35.rename("code 35"), succ.has_36.rename("code 36")]))

The trials tables still hold only times and one tag ('initial cue display'). 09/13/2024 still has no code
100, and codes 35 and 36 keep the labels 'right cue reward condition 1' and '2', while code 35 marks all 164
successful forced trials with the big reward code and code 36 the other 139. Still present.

## Finding 5

Earlier statement: "The raw FSCV current is labelled amperes, but its values range from -1527 to 1557;
the unit is likely nA or the data are not current."

In [ ]:
d = F["shape"]["acquisition/fscv_response_series/data"]
x = d[:2_000_000]
print("unit:", A._str(d.attrs["unit"]), " conversion:", d.attrs["conversion"])
print(f"stored values of the first 2e6 samples: {x.min():.1f} to {x.max():.1f}; times conversion, in nA: "
      f"{x.min() * d.attrs['conversion'] / 1e-9:.1f} to {x.max() * d.attrs['conversion'] / 1e-9:.1f}")

The stored values are in nA with a conversion of 1e-9, so they read as amperes: fixed.

## Finding 6

Earlier statement: "The raw FSCV scan times are an exact 0.1 s grid starting at the first trial start and
running 24120 s, past the last trial, and the raw current cannot be matched to the trial-aligned oxidation
current at any offset or to the task at the stored times. The conversion notes say the times were
interpolated between trial-start codes; the data do not support that they are on the trial clock."

In [ ]:
t_scan, cur, volt, dt = A.fscv_scans(F["shape"])
starts = np.sort(EV["shape"][A.TRIAL_START])
print(f"scans: {len(t_scan)}, from {t_scan[0]:.2f} to {t_scan[-1]:.2f} s; first trial start {starts[0]:.2f} s, "
      f"last {starts[-1]:.2f} s")
print("intervals between scan starts (s):", np.unique(np.round(np.diff(t_scan), 4)))
T = A.fscv_table(F["shape"])
j06 = int(np.argmin(np.abs(volt[:int(np.argmax(volt))] - 0.6)))
raw = cur[:, j06].astype(float)
# per row of the trial-aligned table: r between the scan-to-scan change of the raw current at 0.6 V and
# that of the stored iox, with the raw scans taken at the row's times and shifted by a number of scans
for lag in [-50, -2, -1, 0, 1, 2, 50]:
    rs = []
    for r in range(len(T["start_time"])):
        k = np.searchsorted(t_scan, T["start_time"][r] + np.arange(T["iox"].shape[1]) / A.FSCV_RATE - 0.05) + lag
        a, b = np.diff(raw[k]), np.diff(T["iox"][r])
        ok = np.isfinite(a) & np.isfinite(b)
        rs.append(np.corrcoef(a[ok], b[ok])[0, 1])
    rs = np.array(rs)
    print(f"raw current at {volt[j06]:.3f} V, shift {lag:+d} scans: median r over {len(rs)} rows {np.median(rs):.3f}, "
          f"rows with r > 0 {np.mean(rs > 0):.3f}")

The scans no longer start at the first trial start. Per row the correlation with the stored iox is small,
but it is positive in most rows at shift 0 and falls to about half the rows 50 scans away, so the raw
current is on the trial clock: fixed.

## Finding 7

Earlier statement: "The bipolar LFP of session 09/26/2024 stores values already in volts (around 1e-4)
with a conversion factor of 3.05e-8, so that read as NWB prescribes it is in picovolts."

In [ ]:
lfp = F["direction"]["processing/ecephys/FilteredEphys/bipolar_lfp_series"]
x = lfp["data"][:2_000_000, 0]
print("unit:", A._str(lfp["data"].attrs["unit"]), " conversion:", lfp["data"].attrs["conversion"])
print(f"median |value| of the first 2e6 samples {np.median(np.abs(x)):.3g}, 99.9th percentile {np.percentile(np.abs(x), 99.9):.3g}")

The conversion is now 1 and the values are in volts: fixed.

## Finding 8

Earlier statement: "The raw ephys of session 09/26/2024 has starting_time 0 and no timestamps, but it
matches the stored LFP (r 0.92 to 0.98) only when read 9831.35 s later on the trial clock, consistently
from the first trial to the last."

In [ ]:
es = F["direction"]["acquisition/electrical_series"]
t0_raw, rate = es["starting_time"][()], es["starting_time"].attrs["rate"]
t_lfp = lfp["timestamps"][:]
print(f"raw starting_time {t0_raw:.3f} s at {rate:.4f} Hz; LFP from {t_lfp[0]:.3f} s")
b, a = signal.butter(4, [13, 28], btype="bandpass", fs=1000)
for T0 in [8000.0, 22000.0]:
    i, n = int((T0 - t0_raw) * rate), int(60 * rate)
    r = es["data"][i:i + n, :].astype(float) * es["data"].attrs["conversion"]
    tr_ = t0_raw + (i + np.arange(n)) / rate
    bl = signal.sosfiltfilt(signal.butter(4, 400, fs=rate, output="sos"), r[:, 0] - r[:, 1])
    grid = np.arange(T0 + 5, T0 + 55, 0.001)
    il = np.searchsorted(t_lfp, T0 - 1)
    jl = np.searchsorted(t_lfp, T0 + 61)
    rf = signal.filtfilt(b, a, np.interp(grid, tr_, bl))
    lf = signal.filtfilt(b, a, np.interp(grid, t_lfp[il:jl], lfp["data"][il:jl, 0]))
    N, lg = len(rf), np.arange(-3000, 3001)
    c = np.array([np.corrcoef(rf[3000 + L:N - 3000 + L], lf[3000:N - 3000])[0, 1] for L in lg])
    print(f"segment at {T0:.0f} s: raw c3bs - c3a against the LFP, 13-28 Hz, best lag {lg[np.argmax(c)]} ms "
          f"(r {c.max():.3f}), r at lag 0 {c[3000]:.3f}")

The raw series now starts on the session clock and matches the LFP near lag 0: fixed.

## Finding 9

Earlier statement: "In Monkey P's file the spike times (6800 to 18785 s) are 6800.666 s later than the
raw signal's timestamps (0 to 11985 s); the spike-triggered average of the raw signal shows the spike only
with that shift. The file does not say the two use different clocks."

In [ ]:
p = F["monkey_p"]
esp = p["acquisition/electrical_series"]
u = p["units"]
ui, ust = u["spike_times_index"][:], u["spike_times"][:]
units = {A._str(nm): ust[(ui[j - 1] if j else 0):ui[j]] for j, nm in enumerate(u["unit_name"][:])}
print(f"raw timestamps {esp['timestamps'][0]:.3f} to {esp['timestamps'][-1]:.3f} s")
for k, v in units.items():
    print(f"unit {k}: {v.min():.3f} to {v.max():.3f} s")
# spike-triggered average of the 300-6000 Hz raw signal over the first 300 s, at shifts of the spike index
n = 9_000_000
r = esp["data"][:n, 0].astype(float) * esp["data"].attrs["conversion"]
ts = esp["timestamps"][:n]
y = signal.sosfiltfilt(signal.butter(4, [300, 6000], btype="bandpass", fs=30000, output="sos"), r)
sp = units["ch33#1"]
ii = np.searchsorted(ts, sp[(sp > ts[100]) & (sp < ts[-100])])
sta = np.mean([y[i - 30:i + 41] for i in ii], axis=0)
ctl = np.mean([y[i - 30 + 15000:i + 41 + 15000] for i in ii if i + 15041 < n], axis=0)
print(f"{len(ii)} spikes of ch33#1 in the first 300 s; spike-triggered average from -30 to +40 samples: trough "
      f"{sta.min() * 1e6:.1f} uV at sample {int(np.argmin(sta)) - 30} from the spike time; with the spike times moved "
      f"0.5 s later, range {ctl.min() * 1e6:.1f} to {ctl.max() * 1e6:.1f} uV")

Spikes and raw signal now share one clock: fixed.

## Finding 10

Earlier statement: "In Monkey P's units table, unit 0 merges the unsorted (0) and poor-quality (-1)
thresholded spikes, including 7 spike times below 1 s and 3 repeated times; the poor-quality spikes of the
thresholded table are not in time order."

In [ ]:
tu = p["processing/ecephys/thresholded_units"]
ti, tst = tu["spike_times_index"][:], tu["spike_times"][:]
thr = {A._str(nm): tst[(ti[j - 1] if j else 0):ti[j]] for j, nm in enumerate(tu["unit_name"][:])}
for name, tab in [("units", units), ("thresholded", thr)]:
    for k, v in tab.items():
        print(f"{name} {k}: {len(v)} spikes, below 1 s {int((v < 1).sum())}, repeated {len(v) - len(np.unique(v))}, "
              f"in time order {bool(np.all(np.diff(v) >= 0))}")
print("ch33#0 holds thresholded -1 and 0:", np.array_equal(units["ch33#0"], np.sort(np.r_[thr["-1"], thr["0"]])))
print("units description:", A._str(u.attrs["description"]))

Unit ch33#0 still holds the unsorted and poor-quality spikes, and the units table's description now says so;
no spike time is below 1 s or repeated, and every unit is in time order: fixed.

## Finding 13, the part not reported by amjad-2025-02

Earlier statement: "Monkey P's site is named cl3 in the file and dandiset description, c33 in the Figure 5
legend; the files' related_publications cite the bioRxiv preprint, not the article; the acquisition high
cut is 8000 Hz in the Neuralynx settings, not the 7500 Hz the description states."

In [ ]:
for k, f in F.items():
    print(k, [A._str(x) for x in f["general/related_publications"][:]])

All three files now cite the article (doi:10.1021/acschemneuro.5c00071) as well as the preprint. The site
name and the high cut are reported by amjad-2025-02 (its findings 6 and 4).